In [ ]:
import os
import glob
import torch
from torch.utils.data import Dataset, DataLoader

class SpatialTranscriptomicsDataset(Dataset):
    def __init__(self, data_dir: str):
        """
        参数:
            data_dir (str): 存放 .pt 文件的目录路径
        """
        super().__init__()
        self.data_dir = data_dir
        # 查找目录下所有的 .pt 文件
        self.file_paths = glob.glob(os.path.join(data_dir, "*.pt"))
        
        if len(self.file_paths) == 0:
            print(f"警告：在 {data_dir} 中没有找到任何 .pt 文件！")

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        # 1. 加载单个 tile 的字典数据
        file_path = self.file_paths[idx]
        tile_data = torch.load(file_path, weights_only=True)
        
        # 2. 提取稀疏张量并转换为稠密张量 (Dense)
        # 原始形状为 (H, W, C) 或 (H, W, 1)
        input_expr = tile_data["input_expr"].to_dense()
        input_nuclei = tile_data["input_nuclei"].to_dense()
        target_expr = tile_data["target_expr"].to_dense()
        target_cell_id = tile_data["target_cell_id"].to_dense()
        
        # 3. 调整维度顺序从 (H, W, C) 变为 (C, H, W)
        # 这样才符合 PyTorch 卷积层 (Conv2d) 的输入要求
        input_expr = input_expr.permute(2, 0, 1)
        input_nuclei = input_nuclei.permute(2, 0, 1).float() # Mask 通常转为 float 以便拼接或计算
        target_expr = target_expr.permute(2, 0, 1)
        target_cell_id = target_cell_id.permute(2, 0, 1).float()
        
        # 返回处理好的稠密张量字典
        return {
            "input_expr": input_expr,
            "input_nuclei": input_nuclei,
            "target_expr": target_expr,
            "target_cell_id": target_cell_id
        }


In [ ]:
# ==========================================
# 使用示例 (DataLoader)
# ==========================================
if __name__ == "__main__":
    # 1. 定义数据目录 (替换为你实际的保存路径)
    DATA_DIR = '/root/autodl-tmp/dataset/NSCLC'
    
    # 2. 实例化 Dataset
    dataset = SpatialTranscriptomicsDataset(data_dir=DATA_DIR)
    print(f"总共加载了 {len(dataset)} 个样本。")
    
    # 3. 创建 DataLoader
    # batch_size: 每次喂给模型的样本数
    # shuffle: 训练时打乱数据顺序
    # num_workers: 多线程加载数据，加速读取
    dataloader = DataLoader(
        dataset, 
        batch_size=4, 
        shuffle=True, 
        num_workers=0, 
        pin_memory=True # 如果使用 GPU 训练，开启此项可加速数据传输
    )
    
    # 4. 测试读取一个 Batch 的数据
    for batch_idx, batch_data in enumerate(dataloader):
        print(f"Batch {batch_idx}:")
        print(f" - input_expr shape: {batch_data['input_expr'].shape}")
        print(f" - input_nuclei shape: {batch_data['input_nuclei'].shape}")
        print(f" - target_expr shape: {batch_data['target_expr'].shape}")
        print(f" - target_cell_id shape: {batch_data['target_cell_id'].shape}")
        break # 只打印第一个 batch 用于测试